# LeetCode 984 — String Without AAA or BBB bằng Python

Link đề: https://leetcode.com/problems/string-without-aaa-or-bbb/ · Độ khó: **Medium** · Chủ đề: String (chuỗi), Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        ...
```

Ghi chú: cho `a` và `b`, trả về **một** chuỗi bất kỳ có đúng `a` chữ `'a'`, đúng `b` chữ `'b'`, **không** chứa chuỗi con `"aaa"` và **không** chứa `"bbb"`. Đề bảo đảm luôn tồn tại. Vì có **nhiều đáp án đúng** (ví dụ `a = 1, b = 2` thì `"abb"`, `"bab"`, `"bba"` đều đúng), ta **không so sánh với đáp án mẫu** mà dùng một **hàm kiểm tra tính hợp lệ**: độ dài, số lượng từng chữ, và không có `aaa` / `bbb`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xây chuỗi theo số lượng cho trước, cấm ba chữ giống nhau liên tiếp.
2. Nắm các công cụ Python cần thiết: danh sách ký tự rồi `''.join`, `str.count`, toán tử `in` cho chuỗi con (`'aaa' in s`), `itertools.combinations` cho vét cạn, và đệ quy quay lui.
3. Hiểu nguyên lý **tham lam**: mỗi bước chọn chữ còn **nhiều hơn**, trừ khi hai chữ vừa viết đã giống nhau (khi đó buộc đổi chữ); và nguyên lý **chia khối**: chữ nhiều hơn viết thành khối tối đa 2, ngăn cách bằng một chữ ít hơn.
4. Đi qua 4 cách: vét cạn mọi cách đặt các chữ `a` (oracle, chỉ cho cỡ nhỏ), quay lui sinh từng ký tự, tham lam từng ký tự (cách nộp chính) và dựng chuỗi theo khối; kiểm tra bằng **hàm kiểm tra tính hợp lệ** chứ không so sánh với đáp án mẫu.
5. Tránh các lỗi kinh điển: luôn chọn chữ nhiều hơn mà quên giới hạn **hai chữ liên tiếp**, truy cập `res[-2]` khi chuỗi còn ngắn, khối `"aab"` không chặn bằng `min` khi hết chữ, và **so sánh chuỗi trả về với đáp án mẫu** (bỏ qua các đáp án đúng khác).

## 2. Đề bài & ví dụ

Cho hai số nguyên `a` và `b`, trả về **một** chuỗi `s` bất kỳ sao cho:

- `s` có độ dài `a + b` và chứa **đúng** `a` chữ `'a'` và **đúng** `b` chữ `'b'`,
- chuỗi con `'aaa'` **không** xuất hiện trong `s`,
- chuỗi con `'bbb'` **không** xuất hiện trong `s`.

**Ví dụ 1:** `a = 1, b = 2` -> `"abb"` (`"abb"`, `"bab"` và `"bba"` đều là đáp án đúng).

**Ví dụ 2:** `a = 4, b = 1` -> `"aabaa"`.

**Ràng buộc:**

- `0 <= a, b <= 100`
- Đề **bảo đảm** tồn tại một chuỗi `s` như vậy với `a`, `b` đã cho.

In [1]:
from typing import List, Optional, Tuple
from itertools import combinations
from functools import lru_cache
import random, time, math

def is_valid(s: str, a: int, b: int) -> bool:
    """Hàm kiểm tra: chỉ gồm 'a'/'b', đúng a chữ 'a' và b chữ 'b', không có 'aaa' và không có 'bbb'."""
    return (isinstance(s, str) and set(s) <= {"a", "b"} and len(s) == a + b
            and s.count("a") == a and s.count("b") == b
            and "aaa" not in s and "bbb" not in s)

examples = [(1, 2, "abb"), (4, 1, "aabaa")]
for a, b, sample in examples:
    print(f"a={a} b={b} đáp án mẫu={sample!r} -> hợp lệ: {is_valid(sample, a, b)}")
    assert is_valid(sample, a, b)
print("đáp án khác cũng được:", [s for s in ("abb", "bab", "bba") if is_valid(s, 1, 2)])
print("đáp án sai:", is_valid("aabab", 3, 2), is_valid("abbb", 1, 3), is_valid("ab", 1, 2), is_valid("abc", 1, 1))

a=1 b=2 đáp án mẫu='abb' -> hợp lệ: True
a=4 b=1 đáp án mẫu='aabaa' -> hợp lệ: True
đáp án khác cũng được: ['abb', 'bab', 'bba']
đáp án sai: True False False False


## 3. Kiến thức nền

### 3.1. Kiểm tra chuỗi con và đếm ký tự

`'aaa' in s` kiểm tra `s` có chứa chuỗi con `aaa` không; `s.count('a')` đếm số chữ `a` (không chồng lấn). Chuỗi hợp lệ cần đúng cả ba điều kiện của đề; `is_valid` ở trên gom chúng lại.

In [2]:
s = "aabaa"
print("aaa trong s?", "aaa" in s, "| bbb trong s?", "bbb" in s, "| count a, b:", s.count("a"), s.count("b"))
print("aaaa.count('aa') =", "aaaa".count("aa"), "(không chồng lấn); 'aaa' in 'baaab' =", "aaa" in "baaab")

aaa trong s? False | bbb trong s? False | count a, b: 4 1
aaaa.count('aa') = 2 (không chồng lấn); 'aaa' in 'baaab' = True


### 3.2. Dựng chuỗi bằng danh sách

Ghép chuỗi bằng `+=` nhiều lần tạo chuỗi mới mỗi lần; cách chuẩn là gom vào **danh sách** rồi `"".join(...)` một lần. Truy cập phần tử cuối `res[-1]` và áp cuối `res[-2]` chỉ an toàn khi `len(res) >= 2`.

In [3]:
res = []
for ch in "abba":
    res.append(ch)
print("".join(res), "| res[-1] =", res[-1], "| res[-2] =", res[-2])
try:
    empty = []
    empty[-1]
except IndexError as e:
    print("res[-1] trên danh sách rỗng:", type(e).__name__)

abba | res[-1] = a | res[-2] = b
res[-1] trên danh sách rỗng: IndexError


### 3.3. Hàm kiểm tra thay vì so sánh đáp án

Có nhiều chuỗi đúng cho cùng `(a, b)`, nên test phải dùng `is_valid(s, a, b)`, **không** `s == đáp_án_mẫu`. Hàm kiểm tra này **độc lập** với mọi thuật toán giải.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

**Khi nào tồn tại?** Mỗi chữ `b` chia chuỗi thành các đoạn toàn `a` có độ dài `<= 2`. Với `b` chữ `b` có tối đa `b + 1` đoạn `a`, nên cần `a <= 2 * (b + 1)`. Tương tự `b <= 2 * (a + 1)`. Đề bảo đảm hai điều kiện này (và ta sẽ kiểm chứng bằng máy ở mục 10).

Ví dụ 2: `a = 4, b = 1`. Một chữ `b` chia chuỗi thành hai đoạn `a`, mỗi đoạn `<= 2`: `"aa" + "b" + "aa"`.

**Ý tưởng tham lam:** nhiều chữ `a` hơn thì ưu tiên viết `a` (để không dồn ứ), nhưng nếu hai chữ vừa viết đều là `a` thì **bắt buộc** viết `b`.

Ví dụ 1 (`a = 1, b = 2`): `b` nhiều hơn -> `b`; còn `(1, 1)`, `a >= b` -> `a`; còn `(0, 1)` -> `b`: `"bab"`. Hợp lệ (khác đáp án mẫu `"abb"` nhưng vẫn đúng).

## 5. Cách 1 — Brute Force: thử mọi cách đặt các chữ `a`

**Ý tưởng.** Chuỗi gồm `a + b` vị trí; chọn `a` vị trí để đặt chữ `a`, các vị trí còn lại là `b`. Với mỗi cách chọn (`itertools.combinations(range(n), a)`), dựng chuỗi và kiểm tra `is_valid`; trả về chuỗi hợp lệ đầu tiên. Không dùng bất kỳ ý tưởng tham lam nào, nên đây là **oracle độc lập**.

In [4]:
def str_brute(a: int, b: int) -> str:
    n = a + b
    for pos in combinations(range(n), a):
        chars = ["b"] * n
        for p in pos:
            chars[p] = "a"
        s = "".join(chars)
        if "aaa" not in s and "bbb" not in s:
            return s
    return ""                                        # không tồn tại (ngoài bảo đảm của đề)

for a, b, sample in examples:
    got = str_brute(a, b)
    print(f"a={a} b={b}: brute = {got!r}", "hợp lệ" if is_valid(got, a, b) else "SAI")
    assert is_valid(got, a, b)

a=1 b=2: brute = 'abb' hợp lệ
a=4 b=1: brute = 'aabaa' hợp lệ


**Độ phức tạp và nút thắt.** `C(a + b, a)` cách chọn, mỗi cách tốn `O(n)`: với `a = b = 100` là `C(200, 100) ~ 9 * 10^58`. Đo thử:

In [5]:
for a, b in ((4, 4), (6, 6), (8, 8), (10, 10), (11, 11)):
    t0 = time.perf_counter()
    # đếm thời gian duyệt tối đa 2 * 10^5 tổ hợp đầu tiên (chưa kể chi phí dựng và kiểm tra từng chuỗi)
    cnt = 0
    for pos in combinations(range(a + b), a):
        cnt += 1
        if cnt >= 2 * 10 ** 5:
            break
    print(f"a={a:2} b={b:2}: C({a + b},{a}) = {math.comb(a + b, a):>12}; duyệt {cnt} tổ hợp: {(time.perf_counter() - t0) * 1000:7.1f} ms")
print("a = b = 100: C(200, 100) ~ 9 * 10^58 -> không thể")

a= 4 b= 4: C(8,4) =           70; duyệt 70 tổ hợp:     0.1 ms
a= 6 b= 6: C(12,6) =          924; duyệt 924 tổ hợp:     0.1 ms
a= 8 b= 8: C(16,8) =        12870; duyệt 12870 tổ hợp:     2.0 ms
a=10 b=10: C(20,10) =       184756; duyệt 184756 tổ hợp:    18.5 ms
a=11 b=11: C(22,11) =       705432; duyệt 200000 tổ hợp:    17.8 ms
a = b = 100: C(200, 100) ~ 9 * 10^58 -> không thể


Nút thắt: ta thử rất nhiều chuỗi hiển nhiên sai. Nên **xây từng ký tự** và chọn đúng ngay từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Xây từng ký tự, chọn chữ nhiều hơn trừ khi bị buộc đổi.**

- Nếu hai ký tự cuối đã **giống nhau** thì ký tự tiếp theo **phải khác**.
- Ngược lại viết chữ còn **nhiều hơn** (hoà thì tùy ý).

Trực giác: chữ nhiều hơn là chữ "khó xếp" hơn; viết nó sớm khi còn được phép để không bị dồn vào cuối. Dưới điều kiện đề bảo đảm (`a <= 2(b + 1)` và `b <= 2(a + 1)`), thuật toán này **không bao giờ bị kẹt** (không rơi vào tình huống buộc đổi chữ mà hết chữ kia).

Cách khác: **chia khối**: chữ nhiều hơn viết thành khối **2** chữ, rồi 1 chữ ít hơn, lặp lại.

| Cách | Ý tưởng |
|---|---|
| 2 | Quay lui sinh từng ký tự với ràng buộc "không ba chữ liên tiếp" |
| 3 | Tham lam từng ký tự (nộp) |
| 4 | Chia khối: khối 2 chữ nhiều hơn + 1 chữ ít hơn |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quay lui (backtracking) sinh từng ký tự

**Giải thích từng đoạn.**

- `dfs(a, b, res)`: còn `a` chữ `a`, `b` chữ `b` cần đặt, `res` là chuỗi đang xây.
- Điều kiện dừng: `a == b == 0` thì ghi nhận.
- Thử đặt `'a'` nếu còn `a > 0` và **không** tạo `aaa` (hai ký tự cuối không phải `aa`), tương tự `'b'`; không được thì quay lại.
- Không dùng tham lam, chỉ cắt tỉa theo luật "không ba chữ liên tiếp"; có thể đi vào nhánh cụt rồi quay lại nên chỉ chạy tốt cho cỡ nhỏ-vừa.

In [6]:
def str_backtrack(a: int, b: int) -> str:
    res: List[str] = []

    def dfs(a: int, b: int) -> bool:
        if a == 0 and b == 0:
            return True
        for ch in "ab":
            if ch == "a" and a == 0 or ch == "b" and b == 0:
                continue
            if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
                continue                                  # sẽ tạo ba chữ liên tiếp
            res.append(ch)
            if dfs(a - (ch == "a"), b - (ch == "b")):
                return True
            res.pop()                                     # quay lui
        return False

    return "".join(res) if dfs(a, b) else ""

for a, b, sample in examples:
    got = str_backtrack(a, b)
    print(f"a={a} b={b}: quay lui = {got!r}", "hợp lệ" if is_valid(got, a, b) else "SAI")
    assert is_valid(got, a, b)

a=1 b=2: quay lui = 'abb' hợp lệ
a=4 b=1: quay lui = 'aabaa' hợp lệ


**Code nộp LeetCode (Cách 2):** (đúng về kết quả nhưng **quá chậm** trên đầu vào lớn vì đi vào nhánh cụt rồi mới lùi lại; chỉ để minh họa, **không** nên nộp)

In [7]:
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        import sys
        sys.setrecursionlimit(10000)
        res = []

        def dfs(a: int, b: int) -> bool:
            if a == 0 and b == 0:
                return True
            for ch in "ab":
                if (ch == "a" and a == 0) or (ch == "b" and b == 0):
                    continue
                if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
                    continue
                res.append(ch)
                if dfs(a - (ch == "a"), b - (ch == "b")):
                    return True
                res.pop()
            return False

        dfs(a, b)
        return "".join(res)

sol_backtrack = Solution().strWithout3a3b
for a, b, _ in examples:
    assert is_valid(sol_backtrack(a, b), a, b)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** tồi nhất cấp mũ vì không có cơ chế dự đoán nhánh cụt (đo ở mục 11: tăng gần gấp đôi mỗi khi tăng `a` thêm vài đơn vị); bộ nhớ `O(a + b)`.

---

### 7.2. Cách 3 — Tham lam từng ký tự (cách nộp chính)

**Giải thích từng đoạn.**

- Lặp khi còn chữ (`a > 0 or b > 0`).
- Nếu có ít nhất 2 ký tự và hai ký tự cuối **giống nhau**: ký tự mới phải là chữ **còn lại** (`'b'` nếu hai cuối là `'a'`, và ngược lại).
- Ngược lại: chọn chữ còn **nhiều hơn** (`'a'` khi `a >= b`, ngược lại `'b'`).
- Cập nhật bộ đếm.

In [8]:
def str_greedy(a: int, b: int) -> str:
    res: List[str] = []
    while a > 0 or b > 0:
        if len(res) >= 2 and res[-1] == res[-2]:
            write = "b" if res[-1] == "a" else "a"        # buộc đổi chữ
        else:
            write = "a" if a >= b else "b"                # chữ còn nhiều hơn
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
    return "".join(res)

for a, b, sample in examples:
    got = str_greedy(a, b)
    print(f"a={a} b={b}: tham lam = {got!r}", "hợp lệ" if is_valid(got, a, b) else "SAI")
    assert is_valid(got, a, b)

a=1 b=2: tham lam = 'bab' hợp lệ
a=4 b=1: tham lam = 'aabaa' hợp lệ


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        res = []
        while a > 0 or b > 0:
            if len(res) >= 2 and res[-1] == res[-2]:
                write = "b" if res[-1] == "a" else "a"    # hai chữ cuối giống nhau: buộc đổi
            else:
                write = "a" if a >= b else "b"            # chữ còn nhiều hơn
            res.append(write)
            if write == "a":
                a -= 1
            else:
                b -= 1
        return "".join(res)

sol_greedy = Solution().strWithout3a3b
for a, b, _ in examples:
    assert is_valid(sol_greedy(a, b), a, b)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `a + b` bước, mỗi bước `O(1)`: thời gian `O(a + b)`, bộ nhớ `O(a + b)` cho đáp án.

---

### 7.3. Cách 4 — Dựng chuỗi theo khối

**Ý tưởng khác.** Nếu `a > b`: viết khối `"aa"` (hoặc `"a"` nếu chỉ còn một chữ `a`) rồi **một** chữ `b` (nếu còn). Nếu `b > a`: đối xứng. Nếu `a == b`: viết `"ab"`. Mỗi khối làm hiệu `|a - b|` giảm đi hoặc giữ nguyên; chuỗi gồm các khối ngăn cách bởi chữ ít hơn.

**Giải thích từng đoạn.**

- `take = min(2, count)`: **bảo vệ** khi chữ nhiều hơn chỉ còn 1 chữ (không viết `"aa"` khi chỉ còn `a = 1`).
- `if minority > 0`: chỉ chèn chữ ít hơn khi còn chữ đó.

In [10]:
def str_blocks(a: int, b: int) -> str:
    out: List[str] = []
    while a > 0 or b > 0:
        if a > b:
            take = min(2, a)
            out.append("a" * take)
            a -= take
            if b > 0:
                out.append("b")
                b -= 1
        elif b > a:
            take = min(2, b)
            out.append("b" * take)
            b -= take
            if a > 0:
                out.append("a")
                a -= 1
        else:
            out.append("ab")
            a -= 1
            b -= 1
    return "".join(out)

for a, b, sample in examples:
    got = str_blocks(a, b)
    print(f"a={a} b={b}: khối = {got!r}", "hợp lệ" if is_valid(got, a, b) else "SAI")
    assert is_valid(got, a, b)

a=1 b=2: khối = 'bba' hợp lệ
a=4 b=1: khối = 'aabaa' hợp lệ


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        out = []
        while a > 0 or b > 0:
            if a > b:
                take = min(2, a)                 # khối chữ nhiều hơn: tối đa 2
                out.append("a" * take)
                a -= take
                if b > 0:                        # một chữ ít hơn làm dấu ngăn
                    out.append("b")
                    b -= 1
            elif b > a:
                take = min(2, b)
                out.append("b" * take)
                b -= take
                if a > 0:
                    out.append("a")
                    a -= 1
            else:
                out.append("ab")                 # bằng nhau: xen kẽ
                a -= 1
                b -= 1
        return "".join(out)

sol_blocks = Solution().strWithout3a3b
for a, b, _ in examples:
    assert is_valid(sol_blocks(a, b), a, b)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(a + b)` thời gian và bộ nhớ.

## 8. TRACE từng bước

### 8.1. Tham lam cho `a = 5, b = 3`

In từng bước: hai ký tự cuối, quyết định và bộ đếm.

In [12]:
def trace_greedy(a, b):
    res = []
    step = 0
    while a > 0 or b > 0:
        last2 = "".join(res[-2:])
        if len(res) >= 2 and res[-1] == res[-2]:
            write, why = ("b" if res[-1] == "a" else "a"), "hai chữ cuối giống nhau -> buộc đổi"
        else:
            write, why = ("a" if a >= b else "b"), f"chữ còn nhiều hơn (a={a}, b={b})"
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
        step += 1
        print(f"bước {step}: cuối={last2!r:5} -> viết {write!r} ({why}); còn a={a}, b={b}; chuỗi = {''.join(res)}")
    return "".join(res)

s = trace_greedy(5, 3)
assert is_valid(s, 5, 3)
print("kết quả:", s)

bước 1: cuối=''    -> viết 'a' (chữ còn nhiều hơn (a=5, b=3)); còn a=4, b=3; chuỗi = a
bước 2: cuối='a'   -> viết 'a' (chữ còn nhiều hơn (a=4, b=3)); còn a=3, b=3; chuỗi = aa
bước 3: cuối='aa'  -> viết 'b' (hai chữ cuối giống nhau -> buộc đổi); còn a=3, b=2; chuỗi = aab
bước 4: cuối='ab'  -> viết 'a' (chữ còn nhiều hơn (a=3, b=2)); còn a=2, b=2; chuỗi = aaba
bước 5: cuối='ba'  -> viết 'a' (chữ còn nhiều hơn (a=2, b=2)); còn a=1, b=2; chuỗi = aabaa
bước 6: cuối='aa'  -> viết 'b' (hai chữ cuối giống nhau -> buộc đổi); còn a=1, b=1; chuỗi = aabaab
bước 7: cuối='ab'  -> viết 'a' (chữ còn nhiều hơn (a=1, b=1)); còn a=0, b=1; chuỗi = aabaaba
bước 8: cuối='ba'  -> viết 'b' (chữ còn nhiều hơn (a=0, b=1)); còn a=0, b=0; chuỗi = aabaabab
kết quả: aabaabab


### 8.2. Chia khối cho `a = 5, b = 3`

In từng khối được thêm.

In [13]:
def trace_blocks(a, b):
    out = []
    while a > 0 or b > 0:
        if a > b:
            take = min(2, a)
            block = "a" * take
            a -= take
            if b > 0:
                block += "b"
                b -= 1
        elif b > a:
            take = min(2, b)
            block = "b" * take
            b -= take
            if a > 0:
                block += "a"
                a -= 1
        else:
            block = "ab"
            a -= 1
            b -= 1
        out.append(block)
        print(f"thêm khối {block!r:6} -> còn a={a}, b={b}; chuỗi = {''.join(out)}")
    return "".join(out)

s = trace_blocks(5, 3)
assert is_valid(s, 5, 3)
print("kết quả:", s)

thêm khối 'aab'  -> còn a=3, b=2; chuỗi = aab
thêm khối 'aab'  -> còn a=1, b=1; chuỗi = aabaab
thêm khối 'ab'   -> còn a=0, b=0; chuỗi = aabaabab
kết quả: aabaabab


### 8.3. Nhiều đáp án đúng cho cùng đầu vào

Bốn cách cho bốn chuỗi **khác nhau** nhưng đều hợp lệ với `a = 1, b = 2`. Đó là lý do test phải dùng hàm kiểm tra.

In [14]:
answers = {name: fn(1, 2) for name, fn in (("vét cạn", str_brute), ("quay lui", str_backtrack), ("tham lam", str_greedy), ("khối", str_blocks))}
print(answers, "| đáp án mẫu của đề: 'abb'")
assert all(is_valid(s, 1, 2) for s in answers.values())
print("số chuỗi khác nhau:", len(set(answers.values())))

{'vét cạn': 'abb', 'quay lui': 'abb', 'tham lam': 'bab', 'khối': 'bba'} | đáp án mẫu của đề: 'abb'
số chuỗi khác nhau: 3


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (tham lam từng ký tự): ngắn gọn, `O(a + b)`.

```python
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        res = []
        while a > 0 or b > 0:
            if len(res) >= 2 and res[-1] == res[-2]:
                write = "b" if res[-1] == "a" else "a"    # hai chữ cuối giống nhau: buộc đổi
            else:
                write = "a" if a >= b else "b"            # chữ còn nhiều hơn
            res.append(write)
            if write == "a":
                a -= 1
            else:
                b -= 1
        return "".join(res)
```

In [15]:
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        res = []
        while a > 0 or b > 0:
            if len(res) >= 2 and res[-1] == res[-2]:
                write = "b" if res[-1] == "a" else "a"    # hai chữ cuối giống nhau: buộc đổi
            else:
                write = "a" if a >= b else "b"            # chữ còn nhiều hơn
            res.append(write)
            if write == "a":
                a -= 1
            else:
                b -= 1
        return "".join(res)

sol = Solution()
for a, b, sample in examples:
    got = sol.strWithout3a3b(a, b)
    print(f"a={a} b={b}: {got!r} (đáp án mẫu {sample!r})", "hợp lệ" if is_valid(got, a, b) else "SAI")
    assert is_valid(got, a, b)

a=1 b=2: 'bab' (đáp án mẫu 'abb') hợp lệ
a=4 b=1: 'aabaa' (đáp án mẫu 'aabaa') hợp lệ


## 10. Vì sao đúng

**(a) Điều kiện tồn tại.** Các chữ `b` chia chuỗi thành tối đa `b + 1` đoạn toàn `a`, mỗi đoạn dài `<= 2` nên `a <= 2 * (b + 1)`; đối xứng `b <= 2 * (a + 1)`. Ngược lại, hai điều kiện này **đủ** để tồn tại chuỗi (thuật toán tham lam bên dưới dựng được).

**(b) Tham lam không bị kẹt.** Ý chính: viết chữ còn nhiều hơn giữ cho số chữ còn lại cân bằng (không để một loại bị dồn đến mức vượt quá `2 * (số chữ kia + 1)`), còn khi hai ký tự cuối giống nhau thì chữ đối diện luôn còn để viết, vì nếu hết chữ đối diện mà vẫn còn chữ kia thì bất đẳng thức tồn tại đã bị vi phạm. Thay vì chứng minh dài dòng, ta **kiểm chứng toàn bộ** miền đề bài: mọi cặp `(a, b)` với `0 <= a, b <= 100` thỏa điều kiện tồn tại (điều kiện này ta xác nhận bằng DP đếm cách) đều cho chuỗi hợp lệ.

**(c) Cách 4 (khối).** Mỗi khối chứa tối đa hai chữ giống nhau và liền kề các khối có chữ khác ở đầu/cuối nên không tạo ba chữ giống liền nhau; số chữ mỗi loại được đếm chính xác nhờ `min(2, ...)` và điều kiện `> 0`.

**(d) Độ phức tạp.** `O(a + b)`.

**Kiểm chứng bằng máy.** Hàm `is_valid` độc lập với thuật toán. Ta (1) dùng **DP đếm** số chuỗi hợp lệ để xác nhận điều kiện tồn tại là **đúng** `a <= 2(b + 1)` và `b <= 2(a + 1)` với `0 <= a, b <= 30`; (2) chạy **mọi** cặp `(a, b)` với `0 <= a, b <= 100` thỏa điều kiện qua các cách nhanh và kiểm bằng `is_valid`; (3) so với vét cạn trên mọi cặp nhỏ.

In [16]:
@lru_cache(None)
def count_valid(a: int, b: int, last: str, run: int) -> int:
    """Số chuỗi hợp lệ còn dựng được khi còn a chữ 'a', b chữ 'b', ký tự cuối là last đã lặp run lần."""
    if a == 0 and b == 0:
        return 1
    total = 0
    if a > 0 and not (last == "a" and run == 2):
        total += count_valid(a - 1, b, "a", run + 1 if last == "a" else 1)
    if b > 0 and not (last == "b" and run == 2):
        total += count_valid(a, b - 1, "b", run + 1 if last == "b" else 1)
    return total

def exists_dp(a: int, b: int) -> bool:
    return count_valid(a, b, "", 0) > 0

mismatch = [(a, b) for a in range(31) for b in range(31) if exists_dp(a, b) != (a <= 2 * (b + 1) and b <= 2 * (a + 1))]
print("cặp (a, b) với 0 <= a, b <= 30 mà DP và công thức khác nhau:", mismatch)
assert mismatch == []

valid_pairs = [(a, b) for a in range(101) for b in range(101) if a <= 2 * (b + 1) and b <= 2 * (a + 1)]
print("số cặp hợp lệ trong [0, 100]^2:", len(valid_pairs))
for a, b in valid_pairs:
    for name, fn in (("greedy", str_greedy), ("blocks", str_blocks), ("sol", sol.strWithout3a3b), ("sol_blocks", sol_blocks)):
        s = fn(a, b)
        assert is_valid(s, a, b), (name, a, b, s)
print("tham lam và khối cho chuỗi hợp lệ trên mọi cặp hợp lệ")

cặp (a, b) với 0 <= a, b <= 30 mà DP và công thức khác nhau: []
số cặp hợp lệ trong [0, 100]^2: 5301


tham lam và khối cho chuỗi hợp lệ trên mọi cặp hợp lệ


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi cách đặt | `O(C(a + b, a) * (a + b))` | `O(a + b)` |
| 2. Quay lui | cấp mũ tồi nhất | `O(a + b)` |
| 3. Tham lam | `O(a + b)` | `O(a + b)` |
| 4. Chia khối | `O(a + b)` | `O(a + b)` |

Đo ở cỡ tối đa (`a = b = 100`, và các cặp sát biên như `(100, 49)`):

In [17]:
sys_limit_cases = [(100, 100), (100, 49), (49, 100), (100, 99), (0, 2), (2, 0), (100, 50)]
for a, b in sys_limit_cases:
    for name, fn in (("tham lam", sol_greedy), ("khối", sol_blocks), ("nộp", sol.strWithout3a3b)):
        t0 = time.perf_counter()
        for _ in range(1000):
            s = fn(a, b)
        dt = (time.perf_counter() - t0) / 1000 * 1e6
        assert is_valid(s, a, b)
        print(f"a={a:3} b={b:3} {name:9}: {dt:8.2f} µs, độ dài {len(s)}")
print("Quay lui (Cách 2) tăng rất nhanh vì phải đi vào nhánh cụt rồi lùi lại (mỗi dòng: a = n, b = 2n):")
for n_ in (6, 8, 10, 12, 14):
    t0 = time.perf_counter()
    s = sol_backtrack(n_, 2 * n_)
    print(f"  quay lui ({n_}, {2 * n_}): {(time.perf_counter() - t0) * 1000:9.1f} ms, hợp lệ: {is_valid(s, n_, 2 * n_)}")
print("  (đo ngoài notebook: (16, 32) ~ 5 giây, (20, 40) > 8 giây; (50, 50) và (100, 100) không kết thúc trong thời gian chờ)")

a=100 b=100 tham lam :    33.22 µs, độ dài 200
a=100 b=100 khối     :     7.44 µs, độ dài 200
a=100 b=100 nộp      :    25.03 µs, độ dài 200
a=100 b= 49 tham lam :    18.60 µs, độ dài 149
a=100 b= 49 khối     :     6.90 µs, độ dài 149
a=100 b= 49 nộp      :    20.46 µs, độ dài 149
a= 49 b=100 tham lam :    18.78 µs, độ dài 149
a= 49 b=100 khối     :     7.74 µs, độ dài 149
a= 49 b=100 nộp      :    19.17 µs, độ dài 149
a=100 b= 99 tham lam :    24.70 µs, độ dài 199
a=100 b= 99 khối     :     7.22 µs, độ dài 199


a=100 b= 99 nộp      :    27.23 µs, độ dài 199
a=  0 b=  2 tham lam :     0.35 µs, độ dài 2
a=  0 b=  2 khối     :     0.27 µs, độ dài 2
a=  0 b=  2 nộp      :     0.36 µs, độ dài 2
a=  2 b=  0 tham lam :     0.32 µs, độ dài 2
a=  2 b=  0 khối     :     0.23 µs, độ dài 2
a=  2 b=  0 nộp      :     0.37 µs, độ dài 2
a=100 b= 50 tham lam :    18.90 µs, độ dài 150


a=100 b= 50 khối     :     7.60 µs, độ dài 150
a=100 b= 50 nộp      :    19.37 µs, độ dài 150
Quay lui (Cách 2) tăng rất nhanh vì phải đi vào nhánh cụt rồi lùi lại (mỗi dòng: a = n, b = 2n):
  quay lui (6, 12):       0.2 ms, hợp lệ: True
  quay lui (8, 16):       1.8 ms, hợp lệ: True
  quay lui (10, 20):      14.2 ms, hợp lệ: True
  quay lui (12, 24):      99.0 ms, hợp lệ: True


  quay lui (14, 28):     726.5 ms, hợp lệ: True
  (đo ngoài notebook: (16, 32) ~ 5 giây, (20, 40) > 8 giây; (50, 50) và (100, 100) không kết thúc trong thời gian chờ)


## 12. Bộ test

Vì có **nhiều đáp án hợp lệ**, mọi test dùng `is_valid` (đúng độ dài, đúng số lượng, không `aaa`/`bbb`), **không** so sánh với đáp án mẫu. Gồm: ví dụ của đề, edge case, kiểm tra chính hàm kiểm tra, phản ví dụ cho các cách sai (tìm bằng tìm kiếm), toàn bộ miền hợp lệ và các tính chất.

In [18]:
def all_solutions():
    return {"greedy": str_greedy, "blocks": str_blocks, "backtrack": str_backtrack,
            "sol_greedy": sol_greedy, "sol_blocks": sol_blocks, "final": sol.strWithout3a3b}

def check_all(a, b, with_backtrack=None, with_brute=None):
    if with_backtrack is None:
        with_backtrack = a + b <= 40
    if with_brute is None:
        with_brute = a + b <= 14
    for name, fn in all_solutions().items():
        if name == "backtrack" and not with_backtrack:
            continue
        s = fn(a, b)
        assert is_valid(s, a, b), (name, a, b, s)
    if with_brute:
        assert is_valid(str_brute(a, b), a, b)

# 12.1 ví dụ đề (đáp án mẫu cũng phải qua hàm kiểm tra; ta KHÔNG đòi trùng chuỗi mẫu)
for a, b, sample in examples:
    assert is_valid(sample, a, b)
    check_all(a, b)
print("12.1 ví dụ đề: OK")

# 12.2 kiểm tra chính hàm kiểm tra (bắt đúng các lỗi)
cases_checker = [
    ("abb", 1, 2, True), ("bab", 1, 2, True), ("bba", 1, 2, True), ("aabaa", 4, 1, True), ("", 0, 0, True),
    ("aaab", 3, 1, False),        # có aaa
    ("abbb", 1, 3, False),        # có bbb
    ("ab", 1, 2, False),          # sai độ dài/số lượng
    ("aab", 1, 2, False),         # sai số lượng a, b
    ("abc", 1, 1, False),         # ký tự lạ
    ("ba" * 3 + "b", 3, 4, True),
    ("aabbaabb", 4, 4, True),
]
for s, a, b, expected in cases_checker:
    assert is_valid(s, a, b) is expected, (s, a, b)
print("12.2 kiểm tra hàm kiểm tra: OK", len(cases_checker), "ca")

# 12.3 edge case (miền hợp lệ)
edge = [
    (0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (1, 1), (2, 1), (1, 2), (2, 2), (3, 1), (1, 3),
    (4, 1), (1, 4), (4, 2), (5, 2), (5, 3), (100, 49), (49, 100), (100, 100), (100, 99), (99, 100),
    (50, 50), (100, 50), (50, 100),
]
for a, b in edge:
    assert a <= 2 * (b + 1) and b <= 2 * (a + 1), (a, b)
    check_all(a, b)
print("12.3 edge case: OK", len(edge), "ca (gồm các cặp sát biên a = 2(b + 1))")

12.1 ví dụ đề: OK
12.2 kiểm tra hàm kiểm tra: OK 12 ca
12.3 edge case: OK 24 ca (gồm các cặp sát biên a = 2(b + 1))


In [19]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên mọi cặp hợp lệ nhỏ; hàm kiểm tra phát hiện sai)
def wrong_always_larger(a, b):                    # luôn chọn chữ nhiều hơn, quên giới hạn hai chữ liên tiếp
    res = []
    while a > 0 or b > 0:
        write = "a" if a >= b else "b"
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
    return "".join(res)

def wrong_alternate_then_dump(a, b):              # xen kẽ abab... rồi dồn phần còn lại về cuối
    res = []
    while a > 0 and b > 0:
        res.append("a"); a -= 1
        res.append("b"); b -= 1
    res.append("a" * a + "b" * b)
    return "".join(res)

def wrong_index_error(a, b):                      # res[-2] mà không kiểm tra len(res) >= 2
    res = []
    while a > 0 or b > 0:
        if res[-1] == res[-2]:
            write = "b" if res[-1] == "a" else "a"
        else:
            write = "a" if a >= b else "b"
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
    return "".join(res)

def wrong_blocks_no_min(a, b):                    # khối "aab" / "abb" không chặn bằng min và không kiểm tra hết chữ
    out = []
    while a > 0 or b > 0:
        if a > b:
            out.append("aab"); a -= 2; b -= 1
        elif b > a:
            out.append("bba"); b -= 2; a -= 1
        else:
            out.append("ab"); a -= 1; b -= 1
    return "".join(out)

def find_counter(wrong, limit=10):
    for a in range(limit + 1):
        for b in range(limit + 1):
            if a <= 2 * (b + 1) and b <= 2 * (a + 1):
                try:
                    s = wrong(a, b)
                except Exception as e:
                    return (a, b, f"ngoại lệ {type(e).__name__}")
                if not is_valid(s, a, b):
                    return (a, b, s)
    return None

counters = {}
for name, fn in (("luôn chọn chữ nhiều hơn", wrong_always_larger), ("xen kẽ rồi dồn về cuối", wrong_alternate_then_dump),
                 ("res[-2] khi chuỗi còn ngắn", wrong_index_error), ("khối không chặn bằng min", wrong_blocks_no_min)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    print(f"{name:28} (a, b) = ({case[0]}, {case[1]}): sai -> {case[2]!r}; tham lam đúng -> {str_greedy(case[0], case[1])!r}")
    check_all(case[0], case[1])
print("12.4 phản ví dụ: OK")

luôn chọn chữ nhiều hơn      (a, b) = (1, 4): sai -> 'bbbab'; tham lam đúng -> 'bbabb'
xen kẽ rồi dồn về cuối       (a, b) = (1, 3): sai -> 'abbb'; tham lam đúng -> 'bbab'
res[-2] khi chuỗi còn ngắn   (a, b) = (0, 1): sai -> 'ngoại lệ IndexError'; tham lam đúng -> 'b'
khối không chặn bằng min     (a, b) = (0, 1): sai -> 'bba'; tham lam đúng -> 'b'
12.4 phản ví dụ: OK


In [20]:
# 12.5 toàn bộ miền hợp lệ [0, 100]^2: tham lam, khối, bản nộp; quay lui và vét cạn trên miền nhỏ
cnt = 0
for a in range(101):
    for b in range(101):
        if a <= 2 * (b + 1) and b <= 2 * (a + 1):
            check_all(a, b, with_backtrack=(a + b <= 20), with_brute=(a + b <= 12))
            cnt += 1
print(f"12.5 toàn bộ {cnt} cặp hợp lệ trong [0, 100]^2: OK")

# 12.6 tính chất: đáp án luôn chứa ít nhất một chữ ngăn giữa các khối; số khối của chữ nhiều hơn >= ceil(max/2)
rng = random.Random(6984)
for _ in range(500):
    b = rng.randint(0, 100)
    lo, hi = max(0, (b + 1) // 2 - 1), min(100, 2 * (b + 1))
    a = rng.randint(lo, hi)
    if not (a <= 2 * (b + 1) and b <= 2 * (a + 1)):
        continue
    s = sol.strWithout3a3b(a, b)
    assert is_valid(s, a, b)
    runs_a = [r for r in s.replace("b", " ").split() if r]
    runs_b = [r for r in s.replace("a", " ").split() if r]
    assert all(len(r) <= 2 for r in runs_a + runs_b)                    # mọi đoạn liên tiếp dài <= 2
    assert len(runs_a) >= -(-a // 2) and len(runs_b) >= -(-b // 2)      # cần ít nhất ceil(a/2) đoạn a
print("12.6 tính chất (mọi đoạn <= 2 chữ liên tiếp, đủ số đoạn): OK")

12.5 toàn bộ 5301 cặp hợp lệ trong [0, 100]^2: OK
12.6 tính chất (mọi đoạn <= 2 chữ liên tiếp, đủ số đoạn): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn chọn chữ nhiều hơn, quên giới hạn hai chữ liên tiếp

Chọn chữ còn nhiều hơn là ý tưởng đúng, nhưng thiếu luật "hai chữ cuối đã giống nhau thì phải đổi" sẽ tạo `aaa`/`bbb`.

In [21]:
a, b, s = counters["luôn chọn chữ nhiều hơn"]
print(f"(a, b) = ({a}, {b}): sai -> {s!r} | đúng ->", Solution().strWithout3a3b(a, b))
assert not is_valid(s, a, b) and is_valid(Solution().strWithout3a3b(a, b), a, b)

(a, b) = (1, 4): sai -> 'bbbab' | đúng -> bbabb


Vì sao sai: chữ nhiều hơn cứ được chọn liên tục sẽ tạo ba chữ liền nhau. **Đúng:** nếu `res[-1] == res[-2]` thì buộc đổi chữ.

### Lỗi 2 — Xen kẽ `abab...` rồi dồn phần còn lại về cuối

Dồn mọi chữ thừa vào cuối tạo một đoạn dài.

In [22]:
a, b, s = counters["xen kẽ rồi dồn về cuối"]
print(f"(a, b) = ({a}, {b}): sai -> {s!r} | đúng ->", Solution().strWithout3a3b(a, b))
assert not is_valid(s, a, b)

(a, b) = (1, 3): sai -> 'abbb' | đúng -> bbab


Vì sao sai: các chữ thừa phải được **rải đều** giữa các chữ kia (tối đa 2 chữ liên tiếp). **Đúng:** tham lam hoặc chia khối.

### Lỗi 3 — Truy cập `res[-2]` khi chuỗi còn ngắn

`res[-1] == res[-2]` ngay từ ký tự đầu tiên gây `IndexError`.

In [23]:
a, b, s = counters["res[-2] khi chuỗi còn ngắn"]
print(f"(a, b) = ({a}, {b}): sai -> {s}")
try:
    wrong_index_error(a, b)
except IndexError as e:
    print("ngoại lệ thật sự:", type(e).__name__, e)

(a, b) = (0, 1): sai -> ngoại lệ IndexError
ngoại lệ thật sự: IndexError list index out of range


Vì sao sai: lúc đầu `res` rỗng hoặc chỉ có 1 phần tử. **Đúng:** `if len(res) >= 2 and res[-1] == res[-2]`.

### Lỗi 4 — Khối `"aab"` không chặn khi hết chữ

Viết cố định `"aab"` mà không dùng `min(2, a)` và không kiểm tra `b > 0` làm số chữ bị âm/thừa.

In [24]:
a, b, s = counters["khối không chặn bằng min"]
print(f"(a, b) = ({a}, {b}): sai -> {s!r} (a:{s.count('a')}, b:{s.count('b')}) | đúng ->", Solution().strWithout3a3b(a, b))
assert not is_valid(s, a, b)

(a, b) = (0, 1): sai -> 'bba' (a:1, b:2) | đúng -> b


Vì sao sai: khi chỉ còn 1 chữ `a`, viết `"aab"` dùng thừa chữ. **Đúng:** `take = min(2, a)` và chỉ thêm chữ ngăn nếu `b > 0`.

### Lỗi 5 — So sánh chuỗi trả về với đáp án mẫu

Test kiểu `assert s == "abb"` bác bỏ các đáp án đúng khác như `"bab"` hay `"bba"`: lời giải **đúng** bị chấm sai.

In [25]:
a, b = 1, 2
mine = Solution().strWithout3a3b(a, b)
print("kết quả của bạn:", mine, "| đáp án mẫu: abb | hợp lệ theo đề:", is_valid(mine, a, b), "| bằng đáp án mẫu:", mine == "abb")
assert is_valid(mine, a, b) and mine != "abb"

kết quả của bạn: bab | đáp án mẫu: abb | hợp lệ theo đề: True | bằng đáp án mẫu: False


Vì sao sai: đề nói "trả về **bất kỳ** chuỗi nào" thỏa điều kiện. **Đúng:** kiểm tra bằng hàm kiểm tra tính hợp lệ (đúng độ dài, số lượng, không `aaa`/`bbb`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "xây chuỗi theo số lượng cho trước, cấm lặp liên tiếp quá `k` lần" ⇒ **tham lam**: viết chữ còn nhiều hơn trừ khi bị buộc đổi; hoặc **chia khối** (khối `k - 1` chữ nhiều hơn + 1 chữ ít hơn).
- Điều kiện tồn tại thường là một bất đẳng thức đếm đoạn: `a <= (k - 1) * (b + 1)`.
- Với bài **nhiều đáp án đúng**: viết hàm kiểm tra tính hợp lệ, không so sánh với đáp án mẫu.
- Trước khi truy cập `res[-1]`, `res[-2]` luôn kiểm tra độ dài.
- Dựng chuỗi bằng danh sách rồi `"".join`.
- Cùng họ: Reorganize String (767), Longest Happy String (1405), Task Scheduler (621).
- Test nên có: `(0, 0)`, một loại chữ (`(2, 0)`), các cặp **sát biên** `a = 2(b + 1)`, `(100, 100)`.

## 15. Biến thể

1. **Cấm `k` chữ giống nhau liên tiếp** (thay `3` bằng `k`): điều kiện `a <= (k - 1)(b + 1)`; tham lam đổi chữ khi `k - 1` ký tự cuối giống nhau.
2. **Ba chữ `a, b, c`** (Longest Happy String, 1405): dùng hàng đợi ưu tiên chọn chữ nhiều nhất không vi phạm.
3. **Đếm số chuỗi hợp lệ** (`count_valid` ở mục 10).
4. Bài liên quan: 767, 1405, 621.

Biến thể 1 (cấm `k` chữ liên tiếp), kiểm bằng hàm kiểm tra và DP đếm tổng quát:

In [26]:
def is_valid_k(s: str, a: int, b: int, k: int) -> bool:
    return (set(s) <= {"a", "b"} and len(s) == a + b and s.count("a") == a and s.count("b") == b
            and "a" * k not in s and "b" * k not in s)

def str_greedy_k(a: int, b: int, k: int) -> str:
    res: List[str] = []
    while a > 0 or b > 0:
        if len(res) >= k - 1 and len(set(res[-(k - 1):])) == 1:
            write = "b" if res[-1] == "a" else "a"
        else:
            write = "a" if a >= b else "b"
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
    return "".join(res)

def exists_k(a: int, b: int, k: int) -> bool:
    @lru_cache(None)
    def can(a, b, last, run):
        if a == 0 and b == 0:
            return True
        if a > 0 and not (last == "a" and run == k - 1) and can(a - 1, b, "a", run + 1 if last == "a" else 1):
            return True
        if b > 0 and not (last == "b" and run == k - 1) and can(a, b - 1, "b", run + 1 if last == "b" else 1):
            return True
        return False
    return can(a, b, "", 0)

for k in (2, 3, 4, 5):
    for a in range(0, 13):
        for b in range(0, 13):
            ok = a <= (k - 1) * (b + 1) and b <= (k - 1) * (a + 1)
            assert exists_k(a, b, k) == ok, (a, b, k)
            if ok:
                assert is_valid_k(str_greedy_k(a, b, k), a, b, k), (a, b, k)
print("biến thể cấm k chữ liên tiếp (k = 2..5, a, b <= 12): tham lam hợp lệ trên mọi cặp tồn tại; công thức khớp DP")
print(str_greedy_k(7, 3, 3), str_greedy_k(9, 3, 4))

biến thể cấm k chữ liên tiếp (k = 2..5, a, b <= 12): tham lam hợp lệ trên mọi cặp tồn tại; công thức khớp DP
aabaabaaba aaabaaabaaab


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi cách đặt chữ `a` | `O(C(a + b, a) * n)` | `O(n)` | Oracle, `n <= 16` |
| 2. Quay lui | Sinh từng ký tự + luật 3 chữ | cấp mũ tồi nhất | `O(n)` | Minh họa |
| 3. Tham lam | Chữ nhiều hơn, buộc đổi khi 2 giống | `O(n)` | `O(n)` | **Cách nộp chính** |
| 4. Chia khối | Khối 2 chữ + 1 chữ ít hơn | `O(n)` | `O(n)` | Cách nhìn cấu trúc |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(is_valid("aabaa", 4, 1), is_valid("aaab", 3, 1), is_valid("abab", 2, 2))
print(is_valid("bba", 1, 2), is_valid("bba", 2, 1))
print(Solution().strWithout3a3b(2, 0), Solution().strWithout3a3b(0, 0) == "")
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba giá trị bool
my_guess_2 = None      # hai giá trị bool
my_guess_3 = None      # chuỗi và bool

### Bài 2 — Tính tay (dễ-vừa)

Chạy tay **Cách 3 (tham lam)** với `a = 5, b = 3` và ghi chuỗi thu được (đúng theo thuật toán trong notebook: hoà thì chọn `'a'`).

In [28]:
# Bài 2: điền kết quả tính tay (một chuỗi 8 ký tự)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành tham lam.

In [29]:
def exercise_3(a: int, b: int) -> str:
    res = []
    while a > 0 or b > 0:
        # TODO 1: nếu có >= 2 ký tự và hai ký tự cuối giống nhau thì write = chữ khác
        # TODO 2: ngược lại write = chữ còn nhiều hơn ('a' khi a >= b)
        # TODO 3: thêm write vào res và giảm bộ đếm tương ứng
        pass
    return "".join(res)


# for a, b, _ in examples:
#     assert is_valid(exercise_3(a, b), a, b)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(a, b)` bằng **chia khối** (khối `min(2, ...)` chữ nhiều hơn rồi một chữ ít hơn).

In [30]:
def exercise_4(a: int, b: int) -> str:
    # TODO
    pass


# for a, b, _ in examples:
#     assert is_valid(exercise_4(a, b), a, b)

### Bài 5 — Mini challenge: có tồn tại không? (khó)

Viết `exercise_5(a, b)` trả về `True` nếu **tồn tại** chuỗi có `a` chữ `'a'`, `b` chữ `'b'`, không `aaa` và không `bbb` (không giả sử đề bảo đảm).

```python
exercise_5(1, 2)    -> True
exercise_5(4, 1)    -> True
exercise_5(5, 1)    -> False    # 1 chữ b chia được tối đa 2 đoạn a, mỗi đoạn <= 2: tối đa 4 chữ a
exercise_5(0, 3)    -> False
exercise_5(0, 2)    -> True
exercise_5(100, 49) -> True
exercise_5(100, 48) -> False
```
Gợi ý: đếm số đoạn tối đa mà chữ kia tạo ra.

In [31]:
def exercise_5(a: int, b: int) -> bool:
    # TODO
    pass


# assert exercise_5(1, 2) is True
# assert exercise_5(4, 1) is True
# assert exercise_5(5, 1) is False
# assert exercise_5(0, 3) is False
# assert exercise_5(0, 2) is True
# assert exercise_5(100, 49) is True
# assert exercise_5(100, 48) is False

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `is_valid` kiểm độ dài, số lượng, không `aaa`/`bbb`.
- Mức 2: `"aaab"` có `aaa`; `"bba"` có 1 chữ `a` và 2 chữ `b`, nên hợp lệ với `(1, 2)` nhưng không với `(2, 1)`.
- Mức 3: dòng 1: `True False True`; dòng 2: `True False`; dòng 3: `aa True`.

**Bài 2**
- Mức 1: tham lam: chữ nhiều hơn, hoà thì `'a'`.
- Mức 2: `a, a` (còn `3, 3`), buộc `b`, rồi `a, a`, buộc `b`, rồi `a`, `b` ...
- Mức 3: lần theo từng bước ở mục 8.1; kết quả `"aabaabab"`.

**Bài 3**
- Mức 1: `if len(res) >= 2 and res[-1] == res[-2]: write = "b" if res[-1] == "a" else "a"`.
- Mức 2: `else: write = "a" if a >= b else "b"`.
- Mức 3: `res.append(write)`; `if write == "a": a -= 1 else: b -= 1`.

**Bài 4**
- Mức 1: nếu `a > b`: `take = min(2, a)`.
- Mức 2: thêm `"a" * take` rồi, nếu `b > 0`, thêm `"b"`.
- Mức 3: đối xứng cho `b > a`; nếu bằng nhau thêm `"ab"`.

**Bài 5**
- Mức 1: `b` chữ `b` tạo tối đa `b + 1` đoạn `a`.
- Mức 2: mỗi đoạn dài `<= 2` nên `a <= 2 * (b + 1)`.
- Mức 3: `return a <= 2 * (b + 1) and b <= 2 * (a + 1)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(is_valid("aabaa", 4, 1), is_valid("aaab", 3, 1), is_valid("abab", 2, 2))
print(is_valid("bba", 1, 2), is_valid("bba", 2, 1))
print(Solution().strWithout3a3b(2, 0), Solution().strWithout3a3b(0, 0) == "")

assert (is_valid("aabaa", 4, 1), is_valid("aaab", 3, 1), is_valid("abab", 2, 2)) == (True, False, True)
assert (is_valid("bba", 1, 2), is_valid("bba", 2, 1)) == (True, False)
assert Solution().strWithout3a3b(2, 0) == "aa" and Solution().strWithout3a3b(0, 0) == ""

True False True
True False
aa True


### Đáp án Bài 2

Các bước (a, b còn lại): `a`(4,3), `a`(3,3), buộc `b`(3,2), `a`(2,2), `a`(1,2), buộc `b`(1,1), `a`(0,1), `b`: chuỗi `"aabaabab"`.

In [33]:
hand_answer_solution = "aabaabab"
assert Solution().strWithout3a3b(5, 3) == hand_answer_solution
assert is_valid(hand_answer_solution, 5, 3)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(a: int, b: int) -> str:
    res = []
    while a > 0 or b > 0:
        if len(res) >= 2 and res[-1] == res[-2]:
            write = "b" if res[-1] == "a" else "a"
        else:
            write = "a" if a >= b else "b"
        res.append(write)
        if write == "a":
            a -= 1
        else:
            b -= 1
    return "".join(res)

for a, b, _ in examples:
    assert is_valid(exercise_3_solution(a, b), a, b)
for a, b in valid_pairs:
    assert is_valid(exercise_3_solution(a, b), a, b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(a: int, b: int) -> str:
    out = []
    while a > 0 or b > 0:
        if a > b:
            take = min(2, a)
            out.append("a" * take)
            a -= take
            if b > 0:
                out.append("b")
                b -= 1
        elif b > a:
            take = min(2, b)
            out.append("b" * take)
            b -= take
            if a > 0:
                out.append("a")
                a -= 1
        else:
            out.append("ab")
            a -= 1
            b -= 1
    return "".join(out)

for a, b, _ in examples:
    assert is_valid(exercise_4_solution(a, b), a, b)
for a, b in valid_pairs:
    assert is_valid(exercise_4_solution(a, b), a, b)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(a: int, b: int) -> bool:
    return a <= 2 * (b + 1) and b <= 2 * (a + 1)

assert exercise_5_solution(1, 2) is True
assert exercise_5_solution(4, 1) is True
assert exercise_5_solution(5, 1) is False
assert exercise_5_solution(0, 3) is False
assert exercise_5_solution(0, 2) is True
assert exercise_5_solution(100, 49) is True
assert exercise_5_solution(100, 48) is False
for a in range(31):
    for b in range(31):
        assert exercise_5_solution(a, b) == exists_dp(a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam từng ký tự)

```python
class Solution:
    def strWithout3a3b(self, a: int, b: int) -> str:
        res = []
        while a > 0 or b > 0:
            if len(res) >= 2 and res[-1] == res[-2]:
                write = "b" if res[-1] == "a" else "a"    # hai chữ cuối giống nhau: buộc đổi
            else:
                write = "a" if a >= b else "b"            # chữ còn nhiều hơn
            res.append(write)
            if write == "a":
                a -= 1
            else:
                b -= 1
        return "".join(res)
```

Công thức nhớ nhanh: tồn tại ⇔ `a <= 2(b + 1)` và `b <= 2(a + 1)`; thuật toán: nhiều hơn thì viết, trừ khi hai chữ cuối đã giống nhau.

### Checklist trước khi submit

- [ ] Có luật "hai chữ cuối giống nhau thì buộc đổi", và đã kiểm tra `len(res) >= 2` trước khi truy cập `res[-2]`?
- [ ] Mặc định viết chữ **còn nhiều hơn** (`a >= b`)?
- [ ] Giảm đúng bộ đếm của chữ vừa viết?
- [ ] (Nếu chia khối) có `min(2, ...)` và kiểm tra chữ ngăn còn `> 0`?
- [ ] Test bằng **hàm kiểm tra tính hợp lệ**, không so sánh với đáp án mẫu?
- [ ] Đã thử: `(0, 0)`, `(2, 0)`, các cặp sát biên `a = 2(b + 1)`, `(100, 100)`?